In [12]:

import os
import shutil
import subprocess
from pathlib import Path

print("=== GPU ===")
subprocess.run(["nvidia-smi"], check=False)

print("\n=== CUDA compiler ===")
subprocess.run(["nvcc", "--version"], check=False)

print("\n=== DISK SPACE ===")
for path in ["/kaggle/working", "/kaggle/temp"]:
    if os.path.exists(path):
        total, used, free = shutil.disk_usage(path)
        print(
            path,
            f"Total: {total / 1024**3:.1f} GB",
            f"Free: {free / 1024**3:.1f} GB"
        )

print("\n=== GPU COUNT ===")
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))


=== GPU ===
Thu Nov 12 04:49:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   66C    P0             29W /   70W |    9317MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------

In [13]:
import os
import re
import shutil
import subprocess
from pathlib import Path

ROOT = Path("/kaggle/temp/llama_setup")
ROOT.mkdir(parents=True, exist_ok=True)

LLAMA_SERVER = None

gpu_info = subprocess.check_output(
    ["nvidia-smi"], text=True
)

match = re.search(r"Driver Version:\s*(\d+)\.(\d+)", gpu_info)

if not match:
    raise RuntimeError("Could not detect NVIDIA driver version.")

driver = (int(match.group(1)), int(match.group(2)))
print("NVIDIA driver:", driver)

if driver >= (570, 15):
    print("Using CUDA 12.8 prebuilt llama.cpp v0.9.0")

    archive = ROOT / "llama-cpp.tar.gz"
    url = (
        "https://github.com/ai-dock/llama.cpp-cuda/"
        "releases/download/v0.9.0/"
        "llama.cpp-v0.9.0-cuda-12.8-amd64.tar.gz"
    )

    subprocess.run(
        ["wget", "-O", str(archive), url],
        check=True
    )

    subprocess.run(
        ["tar", "-xzf", str(archive), "-C", str(ROOT)],
        check=True
    )

    candidates = list(ROOT.rglob("llama-server"))

    if not candidates:
        raise RuntimeError(
            "llama-server was not found after extracting the archive."
        )

    LLAMA_SERVER = str(candidates[0])
    os.chmod(LLAMA_SERVER, 0o755)

else:
    print("Driver is below CUDA 12.8 minimum.")

    if not shutil.which("nvcc"):
        raise RuntimeError(
            "No compatible prebuilt binary selected, and nvcc is "
            "not installed. Stop here and inspect Cell 1 output."
        )

    src = ROOT / "llama.cpp"
    build = src / "build"

    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "--branch", "v0.9.0",
            "https://github.com/ggml-org/llama.cpp.git",
            str(src)
        ],
        check=True
    )

    subprocess.run(
        [
            "cmake", "-S", str(src), "-B", str(build),
            "-DGGML_CUDA=ON",
            "-DCMAKE_CUDA_ARCHITECTURES=75",
            "-DCMAKE_BUILD_TYPE=Release"
        ],
        check=True
    )

    subprocess.run(
        [
            "cmake", "--build", str(build),
            "--target", "llama-server",
            "-j", "4"
        ],
        check=True
    )

    LLAMA_SERVER = str(build / "bin" / "llama-server")

if not Path(LLAMA_SERVER).exists():
    raise RuntimeError("llama-server installation failed.")

print("\nInstalled:", LLAMA_SERVER)
env = os.environ.copy()
env["LD_LIBRARY_PATH"] = str(Path(LLAMA_SERVER).parent) + os.pathsep + env.get("LD_LIBRARY_PATH", "")
subprocess.run([LLAMA_SERVER, "--version"], env=env, check=False)

NVIDIA driver: (580, 178)
Using CUDA 12.8 prebuilt llama.cpp v0.9.0


--2026-11-12 04:49:26--  https://github.com/ai-dock/llama.cpp-cuda/releases/download/v0.9.0/llama.cpp-v0.9.0-cuda-12.8-amd64.tar.gz
Resolving github.com (github.com)... 20.27.177.113
Connecting to github.com (github.com)|20.27.177.113|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/1081773475/b572f903-b71d-4285-acb9-1263fb483bc2?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-11-12T05%3A27%3A19Z&rscd=attachment%3B+filename%3Dllama.cpp-v0.9.0-cuda-12.8-amd64.tar.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-11-12T04%3A27%3A04Z&ske=2026-11-12T05%3A27%3A19Z&sks=b&skv=2018-11-09&sig=1gfiY%2BxlicK0NGXjSk3QgZC09ebuGbgn%2B6Y55LSLn1s%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc5NDQ2MjU0NywibmJmIjox


Installed: /kaggle/temp/llama_setup/cuda-12.8/llama-server


/kaggle/temp/llama_setup/cuda-12.8/llama-server: error while loading shared libraries: libcudart.so.12: cannot open shared object file: No such file or directory


CompletedProcess(args=['/kaggle/temp/llama_setup/cuda-12.8/llama-server', '--version'], returncode=127)

In [14]:

%pip install -q -U huggingface_hub

import shutil
from pathlib import Path
from huggingface_hub import hf_hub_download

MODEL_DIR = Path("/kaggle/temp/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

REPO = "ggml-org/Qwen3-Coder-30B-A3B-Instruct-GGUF"
FILENAME = "Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf"

free_gb = shutil.disk_usage(MODEL_DIR).free / 1024**3
print(f"Available disk: {free_gb:.1f} GB")

if free_gb < 21:
    raise RuntimeError(
        "Insufficient free disk for this 18 GB model and download overhead."
    )

MODEL_PATH = hf_hub_download(
    repo_id=REPO,
    filename=FILENAME,
    local_dir=str(MODEL_DIR)
)

print("Model downloaded:", MODEL_PATH)
print(
    "Model size:",
    f"{Path(MODEL_PATH).stat().st_size / 1024**3:.2f} GiB"
)


Note: you may need to restart the kernel to use updated packages.
Available disk: 949.5 GB
Model downloaded: /kaggle/temp/models/Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf
Model size: 16.75 GiB


In [15]:

import os
import sys
import site
import subprocess
from pathlib import Path

# CUDA 12.8 runtime components
PACKAGES = [
    "nvidia-cuda-runtime-cu12==12.8.90",
    "nvidia-cublas-cu12==12.8.4.1"
]

def find_cuda_library(name):
    found = []

    roots = set(site.getsitepackages() + [site.getusersitepackages()])
    roots.update(p for p in sys.path if p)

    for root in roots:
        root = Path(root)

        if not root.is_dir():
            continue

        try:
            found.extend(
                p for p in root.rglob(name)
                if p.is_file() or p.is_symlink()
            )
        except (PermissionError, OSError):
            pass

    return found

runtime_lib = find_cuda_library("libcudart.so.12")
cublas_lib = find_cuda_library("libcublas.so.12")

print("Existing libcudart:", runtime_lib)
print("Existing libcublas:", cublas_lib)

if not runtime_lib or not cublas_lib:
    print("\nInstalling missing CUDA 12.8 runtime packages...")

    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "--no-deps",
            "--no-cache-dir",
            *PACKAGES
        ],
        check=True
    )

runtime_lib = find_cuda_library("libcudart.so.12")
cublas_lib = find_cuda_library("libcublas.so.12")

if not runtime_lib or not cublas_lib:
    raise RuntimeError(
        "CUDA libraries remain missing after installation."
    )

CUDA_LIB_DIRS = sorted({
    str(p.parent)
    for p in runtime_lib + cublas_lib
})

print("\nCUDA runtime directories:")
for directory in CUDA_LIB_DIRS:
    print(directory)

print("\nCUDA RUNTIME INSTALLATION SUCCESSFUL")


Existing libcudart: [PosixPath('/usr/local/lib/python3.13/dist-packages/nvidia/cuda_runtime/lib/libcudart.so.12')]
Existing libcublas: [PosixPath('/usr/local/lib/python3.13/dist-packages/nvidia/cublas/lib/libcublas.so.12')]

CUDA runtime directories:
/usr/local/lib/python3.13/dist-packages/nvidia/cublas/lib
/usr/local/lib/python3.13/dist-packages/nvidia/cuda_runtime/lib

CUDA RUNTIME INSTALLATION SUCCESSFUL


In [16]:

import os
import time
import secrets
import subprocess
import requests
from pathlib import Path

LLAMA_ROOT = Path(LLAMA_SERVER).resolve().parent

env = os.environ.copy()

existing = env.get("LD_LIBRARY_PATH", "").split(os.pathsep)

all_lib_dirs = [
    str(LLAMA_ROOT),
    *CUDA_LIB_DIRS,
    *existing,
    "/usr/local/nvidia/lib64",
    "/usr/local/cuda/lib64"
]

env["LD_LIBRARY_PATH"] = os.pathsep.join(
    dict.fromkeys(p for p in all_lib_dirs if p)
)

# Verify dynamic linker dependencies
check = subprocess.run(
    ["ldd", LLAMA_SERVER],
    env=env,
    capture_output=True,
    text=True
)

print(check.stdout)

if "not found" in check.stdout or "not found" in check.stderr:
    raise RuntimeError("CUDA dependencies are still missing.")

# Test executable
subprocess.run(
    [LLAMA_SERVER, "--version"],
    env=env,
    check=True
)

# Stop an old instance if one exists
if "server_proc" in globals():
    if server_proc.poll() is None:
        server_proc.terminate()
        server_proc.wait(timeout=20)

API_KEY = secrets.token_urlsafe(32)

LOG_PATH = Path("/kaggle/temp/llama-server.log")
server_log = open(LOG_PATH, "w", buffering=1)

cmd = [
    LLAMA_SERVER,
    "-m", MODEL_PATH,
    "--alias", "qwen3-coder-30b",
    "-c", "32768",
    "-ngl", "99",
    "--split-mode", "layer",
    "--tensor-split", "1,1",
    "-fa", "on",
    "--cache-type-k", "q4_0",
    "--cache-type-v", "q4_0",
    "--parallel", "1",
    "--batch-size", "512",
    "--ubatch-size", "128",
    "--jinja",
    "--host", "0.0.0.0",
    "--port", "8080",
    "--api-key", API_KEY,
    "--no-webui"
]

server_proc = subprocess.Popen(
    cmd,
    stdout=server_log,
    stderr=subprocess.STDOUT,
    env=env,
    start_new_session=True
)

print("Loading Qwen3-Coder...")

ready = False

for attempt in range(360):
    if server_proc.poll() is not None:
        break

    try:
        r = requests.get(
            "http://127.0.0.1:8080/health",
            timeout=3
        )

        if r.status_code == 200:
            ready = True
            break

    except requests.RequestException:
        pass

    time.sleep(5)

if not ready:
    server_log.flush()
    print(LOG_PATH.read_text(errors="replace")[-15000:])
    raise RuntimeError("Server startup failed.")

print("\nQWEN3-CODER SERVER READY")
print("Endpoint: http://127.0.0.1:8080/v1")
print("Model: qwen3-coder-30b")
print("API key:", API_KEY)


	linux-vdso.so.1 (0x00007cc26637b000)
	libllama-server-impl.so => /kaggle/temp/llama_setup/cuda-12.8/libllama-server-impl.so (0x00007cc265c5d000)
	libstdc++.so.6 => /lib/x86_64-linux-gnu/libstdc++.so.6 (0x00007cc2659ce000)
	libgcc_s.so.1 => /lib/x86_64-linux-gnu/libgcc_s.so.1 (0x00007cc2659a0000)
	libc.so.6 => /lib/x86_64-linux-gnu/libc.so.6 (0x00007cc26578d000)
	libllama-common.so.0 => /kaggle/temp/llama_setup/cuda-12.8/libllama-common.so.0 (0x00007cc265246000)
	libmtmd.so.0 => /kaggle/temp/llama_setup/cuda-12.8/libmtmd.so.0 (0x00007cc26501b000)
	libllama.so.0 => /kaggle/temp/llama_setup/cuda-12.8/libllama.so.0 (0x00007cc264ba0000)
	libggml.so.0 => /kaggle/temp/llama_setup/cuda-12.8/libggml.so.0 (0x00007cc264b93000)
	libggml-base.so.0 => /kaggle/temp/llama_setup/cuda-12.8/libggml-base.so.0 (0x00007cc264ac1000)
	libssl.so.3 => /lib/x86_64-linux-gnu/libssl.so.3 (0x00007cc264a15000)
	libcrypto.so.3 => /lib/x86_64-linux-gnu/libcrypto.so.3 (0x00007cc264501000)
	libm.so.6 => /lib/x86_64-lin

version: 0.9.0-dev (build 1, commit cd959ba)
built with GNU 11.4.0 for Linux x86_64


Loading Qwen3-Coder...

QWEN3-CODER SERVER READY
Endpoint: http://127.0.0.1:8080/v1
Model: qwen3-coder-30b
API key: kDgPNDXHOxLaMi5Ldf8M1flAqKu4tlIz5z4R6oo64NA


In [17]:

import requests

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

# Check model discovery
r = requests.get(
    "http://127.0.0.1:8080/v1/models",
    headers=headers,
    timeout=30
)
r.raise_for_status()
print("Models:", r.json())

# Test generation
payload = {
    "model": "qwen3-coder-30b",
    "messages": [
        {
            "role": "user",
            "content": "Write a Python function that adds two integers."
        }
    ],
    "temperature": 0.2,
    "max_tokens": 256,
    "stream": False
}

r = requests.post(
    "http://127.0.0.1:8080/v1/chat/completions",
    headers=headers,
    json=payload,
    timeout=300
)
r.raise_for_status()

print(r.json()["choices"][0]["message"]["content"])
print("\nLOCAL API TEST PASSED")


Models: {'models': [{'name': 'qwen3-coder-30b', 'model': 'qwen3-coder-30b', 'modified_at': '', 'size': '', 'digest': '', 'type': 'model', 'description': '', 'tags': [''], 'capabilities': ['completion'], 'parameters': '', 'details': {'parent_model': '', 'format': 'gguf', 'family': '', 'families': [''], 'parameter_size': '', 'quantization_level': ''}}], 'object': 'list', 'data': [{'id': 'qwen3-coder-30b', 'aliases': ['qwen3-coder-30b'], 'tags': [], 'object': 'model', 'architecture': {'input_modalities': ['text'], 'output_modalities': ['text']}, 'created': 1794459132, 'owned_by': 'llamacpp', 'meta': {'vocab_type': 2, 'n_vocab': 151936, 'n_ctx': 32768, 'n_ctx_train': 262144, 'n_embd': 2048, 'n_params': 30532122624, 'size': 17982447616, 'ftype': 'Q4_K - Medium'}}]}
Here's a simple Python function that adds two integers:

```python
def add_integers(a, b):
    """
    Adds two integers and returns their sum.
    
    Args:
        a (int): First integer
        b (int): Second integer
    
  

In [18]:
import subprocess
import time
import re
import requests
from pathlib import Path

# Stop old tunnel process only; do not stop Qwen
if "cf_proc" in globals():
    if cf_proc.poll() is None:
        cf_proc.terminate()
        cf_proc.wait(timeout=15)

CF_BIN = "/kaggle/temp/cloudflared"
CF_LOG = Path("/kaggle/temp/cloudflared-http2.log")

# Confirm local API is still reachable
health = requests.get(
    "http://127.0.0.1:8080/health",
    timeout=10
)
health.raise_for_status()

# Force HTTP/2 over TCP port 7844
log_file = open(CF_LOG, "w", buffering=1)

cf_proc = subprocess.Popen(
    [
        CF_BIN,
        "tunnel",
        "--no-autoupdate",
        "--protocol", "http2",
        "--loglevel", "debug",
        "--url", "http://127.0.0.1:8080"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

print("Starting Cloudflare HTTP/2 tunnel...")

ENDPOINT_URL = None

for i in range(60):
    if cf_proc.poll() is not None:
        break

    log_text = CF_LOG.read_text(errors="replace")

    matches = re.findall(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        log_text
    )

    if matches:
        ENDPOINT_URL = matches[0]
        break

    time.sleep(2)

if not ENDPOINT_URL:
    print("\n===== CLOUDFLARE ERROR LOG =====")
    print(CF_LOG.read_text(errors="replace")[-15000:])

    raise RuntimeError("HTTP/2 tunnel could not establish connection.")

print("\nTUNNEL CREATED:", ENDPOINT_URL)

print("Waiting for DNS propagation...")

for attempt in range(10):
    try:
        response = requests.get(
            ENDPOINT_URL + "/v1/models",
            headers={"Authorization": f"Bearer {API_KEY}"},
            timeout=10
        )
        response.raise_for_status()
        print("PUBLIC API CONNECTED SUCCESSFULLY")
        print("Base URL:", ENDPOINT_URL + "/v1")
        break
    except requests.RequestException:
        print(f"DNS not propagated yet, retrying in 5 seconds...")
        time.sleep(5)
else:
    raise RuntimeError("Public API connection failed after multiple retries.")

Starting Cloudflare HTTP/2 tunnel...

TUNNEL CREATED: https://lower-hardwood-served-merge.trycloudflare.com
Waiting for DNS propagation...
DNS not propagated yet, retrying in 5 seconds...
DNS not propagated yet, retrying in 5 seconds...
PUBLIC API CONNECTED SUCCESSFULLY
Base URL: https://lower-hardwood-served-merge.trycloudflare.com/v1


In [19]:

import requests
import time

url = ENDPOINT_URL + "/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

payload = {
    "model": "qwen3-coder-30b",
    "messages": [
        {
            "role": "user",
            "content": (
                "Write a synthesizable Verilog module for a "
                "32-bit ALU supporting ADD, SUB, AND, OR and XOR. "
                "Include a testable module interface."
            )
        }
    ],
    "temperature": 0.2,
    "max_tokens": 512,
    "stream": False
}

start = time.time()

response = requests.post(
    url,
    headers=headers,
    json=payload,
    timeout=600
)

response.raise_for_status()

result = response.json()
answer = result["choices"][0]["message"]["content"]

print("===== MODEL RESPONSE =====")
print(answer)

print("\n===== PERFORMANCE =====")
print("Elapsed:", round(time.time() - start, 2), "seconds")
print("Finish reason:", result["choices"][0].get("finish_reason"))
print("\nPUBLIC INFERENCE TEST PASSED")


===== MODEL RESPONSE =====
Here's a synthesizable Verilog module for a 32-bit ALU with the requested operations:

```verilog
// 32-bit ALU Module
module alu_32bit (
    input  wire [31:0] A,          // First operand
    input  wire [31:0] B,          // Second operand
    input  wire [2:0]  opcode,        // Operation code
    output wire [31:0] result,         // Output result
    output wire zero_flag                 // Zero flag (set if result is zero)
);

    // Define operation codes
    localparam ADD_OP = 3'b000;
    localparam SUB_OP = 3'b001;
    localparam AND_OP = 3'b010;
    localparam OR_OP  = 3'b011;
    localparam XOR_OP = 3'b100;

    // Internal signal for computation result
    wire [31:0] computed_result;

    // Perform the selected operation
    assign computed_result = 
        (opcode == ADD_OP) ? (A + B) :
        (opcode == SUB_OP) ? (A - B) :
        (opcode == AND_OP) ? (A & B) :
        (opcode == OR_OP)  ? (A | B) :
        (opcode == XOR_OP) ? (A ^ B) : 3